In [1]:
from pathlib import Path
import numpy as np
import json
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Retrouver automatiquement la racine du projet COFI
candidats = [Path.cwd(), *Path.cwd().parents]

RACINE_PROJET = next(
    (
        dossier for dossier in candidats
        if (dossier / "data").is_dir()
        and (dossier / "notebooks").is_dir()
    ),
    None
)

if RACINE_PROJET is None:
    raise FileNotFoundError("Racine du projet COFI introuvable.")

DOSSIER_PROCESSED = RACINE_PROJET / "data" / "processed"
DOSSIER_MODELS = RACINE_PROJET / "models"

DOSSIER_MODELS.mkdir(parents=True, exist_ok=True)

print("Racine du projet :", RACINE_PROJET)
print("Dossier des données :", DOSSIER_PROCESSED)
print("Dossier des modèles :", DOSSIER_MODELS)
print("TensorFlow :", tf.__version__)

I0000 00:00:1793892195.126964  124770 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793892196.422587  124770 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793892199.158201  124770 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Racine du projet : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI
Dossier des données : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/data/processed
Dossier des modèles : /home/nana-marc/Documents/coursBazongo/projetdeep/COFI/models
TensorFlow : 2.21.0


In [2]:
# Chargement des paramètres de préparation
with open(
    DOSSIER_PROCESSED / "parametres_preparation.json",
    "r",
    encoding="utf-8"
) as fichier:
    parametres = json.load(fichier)

# Paramètres du vocabulaire et des séquences
SRC_VOCAB_SIZE = parametres["taille_vocabulaire"]
TGT_VOCAB_SIZE = parametres["taille_vocabulaire"]

MAX_LEN_SRC = parametres["longueur_max_encodeur"]
MAX_LEN_TGT = parametres["longueur_max_cible_complete"]
MAX_LEN_DECODER = parametres["longueur_decodeur"]

# Identifiants des tokens spéciaux
PAD_ID = 0
UNK_ID = 1
SOS_ID = 2
EOS_ID = 3

# Hyperparamètres de l'architecture
EMB_DIM = 128
UNITS = 256

# Reproductibilité
keras.utils.set_random_seed(42)

print("Taille du vocabulaire :", SRC_VOCAB_SIZE)
print("Longueur encodeur :", MAX_LEN_SRC)
print("Longueur cible complète :", MAX_LEN_TGT)
print("Longueur décodeur :", MAX_LEN_DECODER)
print("Tokens spéciaux :", PAD_ID, UNK_ID, SOS_ID, EOS_ID)
print("Dimension Embedding :", EMB_DIM)
print("Unités LSTM :", UNITS)

Taille du vocabulaire : 38125
Longueur encodeur : 60
Longueur cible complète : 62
Longueur décodeur : 61
Tokens spéciaux : 0 1 2 3
Dimension Embedding : 128
Unités LSTM : 256


In [3]:
# Chargement des fichiers préparés
fichier_train = np.load(DOSSIER_PROCESSED / "train_prepared.npz")
fichier_val = np.load(DOSSIER_PROCESSED / "validation_prepared.npz")
fichier_test = np.load(DOSSIER_PROCESSED / "test_prepared.npz")

# Ensemble d'entraînement
X_train = fichier_train["encoder"]
decoder_train = fichier_train["decoder_input"]
Y_train = fichier_train["decoder_target"]

# Ensemble de validation
X_val = fichier_val["encoder"]
decoder_val = fichier_val["decoder_input"]
Y_val = fichier_val["decoder_target"]

# Ensemble de test
X_test = fichier_test["encoder"]
decoder_test = fichier_test["decoder_input"]
Y_test = fichier_test["decoder_target"]

# Vérification des dimensions
print("Entraînement :", X_train.shape, decoder_train.shape, Y_train.shape)
print("Validation  :", X_val.shape, decoder_val.shape, Y_val.shape)
print("Test        :", X_test.shape, decoder_test.shape, Y_test.shape)

Entraînement : (53649, 60) (53649, 61) (53649, 61)
Validation  : (6707, 60) (6707, 61) (6707, 61)
Test        : (6707, 60) (6707, 61) (6707, 61)
